# 07 · Fig. 4 qualitative predictions (inference only, ≈ 5–10 min, any GPU)
Loads the existing seed-0 weights of four run units, predicts one pre-selected test image per unit with the baseline, + clDice, + Skeleton Recall and the baseline with a calibrated stem threshold, and writes class-index PNGs to `analysis/fig4/`. Nothing in `results/`, `splits/` or the data folder is modified. The last cell checks that each prediction reproduces the stored per-image stem IoU.

In [ ]:
# --- Setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/Plant Phenotyping/Claude Research/gwfss_stem_seg'
DATA_ROOT = f'{PROJECT}/content/gwfss'
RESULTS = f'{PROJECT}/results'                    # read only: final.pt and metrics_ss.csv
OUT = f'{PROJECT}/analysis/fig4'                  # the only folder this notebook writes to

import sys, os
sys.path.insert(0, f'{PROJECT}/src')
!pip -q install segmentation-models-pytorch==0.5.0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from gwfss_stem import data
manifest = data.prepare_dataset(DATA_ROOT)   # downloads GWFSS_v1.0 if not present
print(len(manifest), 'images')

In [ ]:
# Images chosen offline by stage4/fig4/select_fig4.py (rule fixed before any prediction was viewed):
# per unit, the test image whose mean stem-IoU change (two losses vs baseline) is closest to the unit median.
# offset = stem-logit offset of the calibrated baseline (random split: validation set; LOIO: lower median of 20 ten-image draws).
ROWS = [
 {
  "split": "random",
  "backbone": "segformer_b2",
  "name": "INRAE_phenomobile_nadir_2023_FFAST_Toulouse_2023-06-23_uplot_23ZM8_Y01X012_camera_1_2_RGB_WB.png",
  "offset": 0.5
 },
 {
  "split": "loio_USASK",
  "backbone": "segformer_b2",
  "name": "USASK_159-compressed_001.png",
  "offset": 3.5
 },
 {
  "split": "loio_UTokyo",
  "backbone": "segformer_b2",
  "name": "UTokyo_20150521_EOS12_IMG_1103.png",
  "offset": 5.0
 },
 {
  "split": "loio_UQ",
  "backbone": "upernet_convnext_t",
  "name": "UQ_new_4c3f721162134e1c7d7723acba72643ac50c504e.png",
  "offset": 2.5
 }
]

In [ ]:
import json, cv2, numpy as np, pandas as pd, torch
from gwfss_stem.engine import _native_logits
from gwfss_stem.models import build_model
from gwfss_stem.data import SegDataset, STEM, IGNORE

os.makedirs(OUT, exist_ok=True)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
check = []
for i, r in enumerate(ROWS):
    it = SegDataset(DATA_ROOT, [r['name']], train=False)[0]
    gt = it['mask'].numpy().astype(np.uint8)
    rgb = cv2.imread(os.path.join(DATA_ROOT, 'images', r['name']))
    cv2.imwrite(f'{OUT}/row{i}_rgb.png', rgb)
    cv2.imwrite(f'{OUT}/row{i}_gt.png', gt)
    for loss in ['base', 'cldice', 'skelrecall']:
        run = f"{RESULTS}/{r['split']}/{r['backbone']}/{loss}/scale1.0/seed0"
        model = build_model(r['backbone'], pretrained=False).to(dev)
        model.load_state_dict(torch.load(f'{run}/final.pt', map_location=dev))
        model.eval()
        lg = _native_logits(model, it['image'][None].to(dev))
        conds = [(loss, 0.0)] + ([('base_cal', r['offset'])] if loss == 'base' else [])
        for cond, b in conds:
            l2 = lg.clone(); l2[:, STEM] += b
            pred = l2.argmax(1)[0].byte().cpu().numpy()
            cv2.imwrite(f'{OUT}/row{i}_{cond}.png', pred)
            valid = gt != IGNORE; p = (pred == STEM) & valid; g = (gt == STEM) & valid
            iou = (p & g).sum() / max((p | g).sum(), 1)
            ref = np.nan
            if b == 0.0:   # must reproduce the stored per-image result
                m = pd.read_csv(f'{run}/metrics_ss.csv').set_index('name').loc[r['name']]
                ref = m.inter_stem / max(m.union_stem, 1)
            check.append(dict(row=i, split=r['split'], backbone=r['backbone'], name=r['name'], cond=cond,
                              offset=b, stem_iou=round(100 * iou, 2), stored_stem_iou=round(100 * ref, 2)))
        del model; torch.cuda.empty_cache()
ck = pd.DataFrame(check); ck.to_csv(f'{OUT}/fig4_check.csv', index=False)
print(ck.to_string())
d = (ck.stem_iou - ck.stored_stem_iou).abs().max()
print(f'max |recomputed - stored| stem IoU = {d:.3f} points  ->', 'OK' if d < 0.5 else 'CHECK')